In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(api_key=groq_api_key, model="Llama3-8b-8192")

In [3]:
from langchain_openai import OpenAIEmbeddings

In [4]:
from langchain_chroma import Chroma

In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
os.environ["HF_TOKEN"]= os.getenv("HF_TOKEN")
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [7]:
import langchain
print(langchain.__version__)
print(langchain.__file__)

1.2.8
/Users/apple/Desktop/Langchain/venv/lib/python3.11/site-packages/langchain/__init__.py


In [14]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
#from langchain.chains.combine_documents import create_stuff_documents_chain


#### Loading the data from webpage

In [21]:
docs = WebBaseLoader("https://docs.langchain.com").load()

# Split
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = splitter.split_documents(docs)

# Vector DB
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(splits, embeddings)

In [23]:
retriever = vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x133dbd3d0>, search_kwargs={})

In [24]:
#Prompt Template
system_prompt = (
    "You are a helpful AI assistant that helps people find information "
    "about Langchain documentation. Use the following pieces of "
    "context to answer the question at the end. If you don't know the answer, "
    "just say you don't know, don't try to make up an answer. "
    "Answer in a concise manner.\n"
    "{context}\n"
    
)

In [30]:
prompt = ChatPromptTemplate.from_template(
    """Answer the question using the context.

Context:
{context}

Question:
{input}
"""
)

In [31]:
# LLM
llm = OpenAI(model="gpt-4o-mini")

In [37]:
from operator import itemgetter

In [38]:
# ✅ FIXED LCEL RAG PIPELINE
rag_chain = (
    {
        "context": itemgetter("input") | retriever,
        "input": itemgetter("input"),
    }
    | prompt
    | llm
)

In [39]:
rag_chain.invoke({"input": "What is LangChain?"})

' \nAnswer:\nLangChain is a platform for agent engineering that allows teams to build reliable AI agents. It provides frameworks in both Python and TypeScript for quickly getting started with agent development, as well as tools for deep agents, low-level orchestration, and memory management. It is trusted by AI teams at companies like Replit, Clay, and Cloudflare.'